In [ ]:
# @title
import re
import json
import warnings
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ============================================================
# 📂 خواندن ۳ تب فایل اکسل
# ============================================================

FILE = ""  # ← اسم فایلت

df_reviews = pd.read_excel(FILE, sheet_name="Customer_Reviews")
df_competitor = pd.read_excel(FILE, sheet_name="Competitor_Overview")
df_price = pd.read_excel(FILE, sheet_name="Price_Overview")

# ===== ابتدا ببینیم چند ستون داریم =====
print(f"تب نظرات: {len(df_reviews.columns)} ستون → {df_reviews.columns.tolist()}")
print(f"تب رقبا:  {len(df_competitor.columns)} ستون → {df_competitor.columns.tolist()}")
print(f"تب قیمت:  {len(df_price.columns)} ستون → {df_price.columns.tolist()}")

# ===== نرمال‌سازی ستون‌ها =====

# تب نظرات: همیشه ۴ ستون
df_reviews.columns = ["رستوران", "سفارش", "متن_نظر", "ستاره"]
df_reviews["ستاره"] = pd.to_numeric(df_reviews["ستاره"], errors="coerce")

# تب رقبا: ۹ یا ۱۰ ستون — ستون اضافی رو نگه دار
comp_cols = ["رستوران", "امتیاز_اسنپفود", "تعداد_امتیازها",
             "ستاره_۵", "ستاره_۴", "ستاره_۳", "ستاره_۲", "ستاره_۱", "امتیاز_گوگل"]

if len(df_competitor.columns) == len(comp_cols):
    df_competitor.columns = comp_cols
elif len(df_competitor.columns) > len(comp_cols):
    # ستون‌های اضافی رو با نام عمومی بذار
    extra = len(df_competitor.columns) - len(comp_cols)
    comp_cols += [f"ستون_اضافی_{i+1}" for i in range(extra)]
    df_competitor.columns = comp_cols
    print(f"⚠️ {extra} ستون اضافی در تب رقبا: {df_competitor.columns[-extra:].tolist()}")
elif len(df_competitor.columns) < len(comp_cols):
    comp_cols = comp_cols[:len(df_competitor.columns)]
    df_competitor.columns = comp_cols

# عدد کردن ستون‌های عددی
for col in df_competitor.columns[1:]:
    df_competitor[col] = pd.to_numeric(df_competitor[col], errors="coerce")

# تب قیمت: ۲ یا بیشتر ستون
price_cols = ["رستوران", "میانگین_قیمت", "امتیاز_قیمت"]

if len(df_price.columns) == len(price_cols):
    df_price.columns = price_cols
elif len(df_price.columns) > len(price_cols):
    extra = len(df_price.columns) - len(price_cols)
    price_cols += [f"ستون_اضافی_قیمت_{i+1}" for i in range(extra)]
    df_price.columns = price_cols
elif len(df_price.columns) < len(price_cols):
    price_cols = price_cols[:len(df_price.columns)]
    df_price.columns = price_cols

for col in df_price.columns[1:]:
    df_price[col] = pd.to_numeric(df_price[col], errors="coerce")

print(f"\n✅ تب نظرات:     {len(df_reviews)} نظر از {df_reviews['رستوران'].nunique()} رستوران")
print(f"✅ تب رقبا:      {len(df_competitor)} رستوران — ستون‌ها: {df_competitor.columns.tolist()}")
print(f"✅ تب قیمت:      {len(df_price)} رستوران — ستون‌ها: {df_price.columns.tolist()}")
print(f"\n🏢 برندها: {sorted(df_reviews['رستوران'].unique())}")

# ============================================================
# 📝 کلمات
# ============================================================

STOPWORDS = set([
    "و", "در", "به", "از", "که", "این", "آن", "با", "برای", "تا", "بر", "پس",
    "اما", "اگر", "یا", "نه", "ولی", "چون", "بین", "همچنین", "بلکه",
    "است", "بود", "شد", "می", "شود", "شده", "کرد", "کند",
    "داشتم", "دارم", "دارند", "داشتن", "داشته",
    "کردم", "کردی", "کردیم", "کردند", "کرده",
    "کنم", "کنی", "کنیم", "کنند", "کنه",
    "شدم", "شدی", "شدیم", "شدند",
    "بودم", "بودی", "بودیم", "بودند", "بوده",
    "میشه", "بشه", "میره", "بره", "بیاد", "باید",
    "هست", "هستند", "هستم", "هستی", "هستیم",
    "ما", "شما", "آنها", "او", "من", "تو", "ای",
    "خودش", "خودم", "خودت", "خودشان",
    "ها", "های", "ی", "ترین", "تر",
    "خیلی", "بسیار", "کمی", "هم", "همه", "دیگر", "اند",
    "همین", "چندین", "چند",
    "الان", "وقت", "وقتی", "بعد", "قبل", "دیگه", "هنوز",
    "چی", "چیزی", "کسی", "چرا", "چطور", "آیا", "مگه",
    "رو", "واسه", "مثل", "دنبال", "دم", "اینکه",
    "آره", "بله", "خب", "ببین",
    "هاش", "هاشو", "هاشون", "شون", "تون", "مون",
    "شاید", "حتماً", "قطعاً",
    "یک", "دو", "سه", "چهار", "پنج",
    "بار", "دفعه", "نفر", "عدد",
    "رفتم", "رفتی", "رفتیم", "رفتند",
    "دادم", "دادی", "دادیم", "دادند",
    "گفتم", "گفتی", "گفتیم", "گفتند",
    "دیدم", "دیدی", "دیدیم", "دیدند",
    "خوردم", "خوردی", "خوردیم", "خوردند",
    "سعی", "تلاش",
    "البته", "واقعاً", "عیناً",
])

POSITIVE_WORDS = set([
    "عالی", "خوب", "خوشمزه", "فوق", "نظیر", "لذیذ", "تازه", "معرکه", "درجه",
    "خوش‌طعم", "دلچسب", "بامزه", "خوش‌مزه", "خوش‌پخت", "خوش‌رنگ", "خوش‌بو",
    "مغزدار", "نرم", "آبدار", "پخته", "سالم", "کامل", "بی‌نقص", "فوق‌العاده",
    "مطبوع", "لذیذترین", "خوشمزه‌ترین",
    "راضی", "رضایت", "رضایتمند", "ممنون", "سپاسگزار", "خوشحال",
    "تکرار", "دوست", "پیشنهاد", "معرفی", "محشر", "گل",
    "حرفه‌ای", "استثنایی", "سوپر", "بی‌نظیر", "خلاقانه", "متفاوت", "خاص", "برتر",
    "آفرین", "عجیب", "مناسب", "رایگان", "قشنگ", "زیبا", "جذاب",
    "عالیست", "خوبه",
    "سریع", "دقیق", "منظم", "مودب", "زود", "فوری", "به‌موقع", "وقت‌شناس",
    "ارزان", "باکیفیت", "ارزش", "حرفه ای", "تخفیف", "پرحجم", "سیرکننده",
    "محکم", "استوار", "ایمن", "مرتب",
    "داغ", "گرم",
])

NEGATIVE_WORDS = set([
    "بد", "بی‌طعم", "بدطعم", "خام", "سوخته", "خشک", "چرب",
    "بیات", "نیم‌پز", "سفت", "بوی بد", "آبکی", "خمیری",
    "شور", "ترش", "تلخ", "تند", "فاسد", "ذخیم",
    "نامطبوع", "پلاسیده", "نپخته", "مانده",
    "سرد",
    "کم", "کوچیک", "ناچیز",
    "ضعیف", "نامناسب", "بی‌کیفیت", "بی‌ارزش", "ناراضی",
    "ناامید", "متأسف", "بدترین", "ضعیف", "افتضاح", "فاجعه",
    "مسخره", "بدی", "کلاهبرداری", "فریب", "گول", "تقلبی",
    "بی کیفیت", "ضایع", "آشغال",
    "گران", "پائین", "پایین", "فراموش", "پرو", "غیرمنطقی",
    "تأخیر", "دیر", "کند", "افت", "معطل", "موندگی", "ناقص",
    "نشت", "تلخ", "لوپیده", "نامرتب", "باز",
    "بی‌ادب", "بدلحن", "بی‌توجهی", "بی‌اعتنایی", "لغو",
    "هیچ", "اصلاً", "هرگز", "ضحم", "شور",
])

ASPECTS = {
    "غذا": [
        "طعم", "خوشمزه", "بدطعم", "بی‌طعم", "لذیذ", "خام", "سوخته",
        "تازه", "بیات", "نیم‌پز", "سفت", "چرب", "خشک", "ذخیم",
        "بامزه", "خوش‌طعم", "خوش‌مزه", "خوش‌پخت", "خوش‌رنگ", "خوش‌بو",
        "شور", "ترش", "نامطبوع", "مطبوع", "ولخته", "آبکی",
        "مغزدار", "نرم", "آبدار", "پخته", "سالم", "فاسد",
        "سرد", "داغ", "گرم", "یخ",
        "حجم", "کم", "زیاد", "پرحجم", "فیله", "ناچیز",
        "برنج", "کباب", "خورش", "جوجه", "گوشت", "ساندویچ",
        "دسر", "سالاد", "نوشیدنی", "نون", "سس", "دورچین",
        "شیشلیک", "کوبیده", "فیله", "ماهیچه", "کتف", "سینه",
        "چلوکباب", "زرشک", "باقالی", "دلمه", "ماست", "شیشلیک",
        "دورچین", "سبزی", "خیار", "فلفل", "کره", "سماق", "گوجه",
    ],
    "ارسال": [
        "تأخیر", "دیر", "زود", "سریع", "کند","به جای", "به جاش", "منتظر", "معطل",
        "تحویل", "فوری", "به‌ موقع", "وقت‌شناس", "عقب",
        "پیک", "دلیوری", "ارسال", "مسیر", "آدرس", "اشتباه", "راننده",
        "رسید", "ناقص", "کامل",
    ],
    "بسته‌بندی": [
        "بسته‌بندی", "پکینگ", "بسته",
        "نشت", "لیزر", "فراموش", "باز",
        "ظرف", "درب", "سلفون", "فویل", "ایربان", "پر حجم",
        "نامرتب", "محکم", "ایمن",
    ],
    "قیمت": [
        "گران", "دو برابر", "قیمت", "ارزش", "توجیه", "قیمت زیاد",
        "حساب", "فاکتور", "تخفیف", "گزاف", "پول", "مبلغ",
        "پول‌سوز", "رایگان", "غیرمنطقی",
        "پر", "پرو", "گول", "کلاه",
        "کم‌فروشی", "گرون", "لوکس",
    ],
}

MIN_MENTIONS = 3

print("\n✅ کلمات آماده")
print(f"   استاپ‌ورد: {len(STOPWORDS)}")
print(f"   مثبت: {len(POSITIVE_WORDS)}")
print(f"   منفی: {len(NEGATIVE_WORDS)}")
for k, v in ASPECTS.items():
    print(f"   جنبه {k}: {len(v)} کلمه")

In [ ]:
# @title
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('ك', 'ک').replace('ي', 'ی')
    text = text.replace('ؤ', 'و').replace('إ', 'ا').replace('أ', 'ا').replace('ة', 'ه')
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    try:
        from hazm import Normalizer
        text = Normalizer().normalize(text)
    except:
        pass
    return text

def tokenize(text):
    try:
        from hazm import WordTokenizer
        return WordTokenizer().tokenize(text)
    except:
        return text.split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def lemmatize(tokens):
    try:
        from hazm import Lemmatizer
        return [Lemmatizer().lemmatize(t) for t in tokens]
    except:
        return tokens

def preprocess(text):
    text = clean_text(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    return tokens

print("✅ توابع پیش‌پردازش آماده")

In [ ]:
# @title
def keyword_sentiment(tokens):
    """امتیاز احساسات فقط از کلمات"""
    pos = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    total = pos + neg
    if total == 0:
        return 0.0, pos, neg
    return round((pos - neg) / total, 3), pos, neg


def star_to_score(star):
    """تبدیل ستاره به امتیاز پایه"""
    if pd.isna(star):
        return 0.0
    star = int(star)
    mapping = {5: 0.9, 4: 0.5, 3: 0.0, 2: -0.5, 1: -0.9}
    return mapping.get(star, 0.0)


def combined_sentiment(star, kw_score):
    """
    ترکیب ستاره (۶۰٪) + کلمات (۴۰٪)

    ستاره = سیگنال صریح مشتری (قوی)
    کلمات = سیگنال ضمنی متن (تکمیلی)
    """
    star_score = star_to_score(star)

    # اگه ستاره نداریم، فقط کلمات
    if pd.isna(star):
        final = kw_score
    else:
        final = 0.7 * star_score + 0.3 * kw_score

    final = max(-1.0, min(1.0, final))

    if final > 0.15:
        label = "مثبت"
    elif final < -0.15:
        label = "منفی"
    else:
        label = "خنثی"

    return label, round(final, 3)


def aspect_analysis(text):
    """تحلیل جنبه‌ای: امتیاز هر جنبه بر اساس کلمات نزدیک"""
    cleaned = clean_text(text)
    tokens = preprocess(cleaned)
    sentences = re.split(r'[.!؟؛\n]', cleaned)

    results = {}
    for aspect_name, keywords in ASPECTS.items():
        # آیا کلمه کلیدی این جنبه تو متن هست؟
        found = [kw for kw in keywords if kw in cleaned]

        if not found:
            results[aspect_name] = None
            continue

        # جملات مرتبط با این جنبه
        relevant = [s for s in sentences if any(kw in s for kw in found)]
        if not relevant:
            relevant = [cleaned]

        # احساسات جملات مرتبط
        scores = []
        for sent in relevant:
            sent_tokens = preprocess(sent)
            sc, _, _ = keyword_sentiment(sent_tokens)
            scores.append(sc)

        if scores:
            avg = np.mean(scores)
            # اگه خیلی صفره (نه مثبت نه منفی)، ولی اشاره شده → ستاره رو بگیر
            results[aspect_name] = round(avg, 3)
        else:
            results[aspect_name] = None

    return results


def analyze_file(df):
    """تحلیل کامل — ستاره + کلمات + جنبه‌ها"""
    results = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="🔬 تحلیل"):
        text = str(row.get("متن_نظر", ""))
        star = row.get("ستاره", None)

        if not text or text == "nan" or len(text) < 3:
            continue

        tokens = preprocess(text)
        kw_score, pos_count, neg_count = keyword_sentiment(tokens)
        label, final_score = combined_sentiment(star, kw_score)
        aspects = aspect_analysis(text)

        # ستاره‌های جنبه‌ای: اگه جنبه null بود ولی اشاره شده
        # از ستاره کمک بگیر
        if pd.notna(star):
            star_aspect_score = star_to_score(star)
            for asp_name, asp_val in aspects.items():
                if asp_val is not None and abs(asp_val) < 0.05:
                    # کلمه جنبه پیدا شد ولی احساسات خنثی بود → ستاره رو وزن بده
                    aspects[asp_name] = round(0.5 * star_aspect_score + 0.5 * asp_val, 3)

        result = {
            "رستوران": row.get("رستوران", ""),
            "سفارش": row.get("سفارش", ""),
            "متن_نظر": text[:150],
            "ستاره": star,
            "احساسات": label,
            "امتیاز_احساسات": final_score,
            "امتیاز_ستاره": star_to_score(star),
            "امتیاز_کلمات": kw_score,
            "کلمات_مثبت": pos_count,
            "کلمات_منفی": neg_count,
        }
        result.update({f"جنبه_{k}": v for k, v in aspects.items()})
        results.append(result)

    df_result = pd.DataFrame(results)

    # خلاصه
    print(f"\n✅ {len(df_result)} نظر تحلیل شد")
    print(f"\n📊 توزیع احساسات:")
    print(df_result["احساسات"].value_counts().to_string())
    print(f"\n📈 میانگین امتیاز: {df_result['امتیاز_احساسات'].mean():+.3f}")

    return df_result


# ===== اجرا =====
df_result = analyze_file(df_reviews)
df_result.to_excel("result_analysis.xlsx", index=False)
df_result.head(30)

In [ ]:
# @title
MIN_MENTIONS = 3

def brand_report(df_result, brand_name):
    bdf = df_result[df_result["رستوران"] == brand_name]
    if bdf.empty:
        return

    total = len(bdf)
    pos = (bdf["احساسات"] == "مثبت").sum()
    neg = (bdf["احساسات"] == "منفی").sum()
    neu = (bdf["احساسات"] == "خنثی").sum()
    avg = bdf["امتیاز_احساسات"].mean()
    avg_star = bdf["ستاره"].mean()

    print(f"\n{'='*55}")
    print(f"  📊 {brand_name}")
    print(f"{'='*55}")
    print(f"  📝 نظرات: {total}")
    print(f"  ⭐ میانگین ستاره: {avg_star:.2f}")
    print(f"  ✅ مثبت: {pos} ({pos/total*100:.1f}%)")
    print(f"  ❌ منفی: {neg} ({neg/total*100:.1f}%)")
    print(f"  ⚪ خنثی: {neu} ({neu/total*100:.1f}%)")
    print(f"  📈 امتیاز احساسات: {avg:+.3f}")
    print()

    for aspect in ASPECTS.keys():
        col = f"جنبه_{aspect}"
        if col in bdf.columns:
            vals = bdf[col].dropna()
            n = len(vals)
            if n < MIN_MENTIONS:
                print(f"  {aspect:>8}: {'░'*20} ⚠️ ناکافی ({n})")
            else:
                m = vals.mean()
                bar_len = int((m + 1) * 10)
                bar = "█" * bar_len + "░" * (20 - bar_len)
                print(f"  {aspect:>8}: {bar} {m:+.2f} ({n})")

    # سفارشات
    orders = bdf["سفارش"].dropna().tolist()
    if orders and str(orders[0]) != "nan":
        counter = Counter()
        for order in orders:
            if isinstance(order, str):
                for item in re.split(r'[،,\+\-و]', order):
                    item = item.strip()
                    if item and len(item) > 1:
                        counter[item] += 1
        if counter:
            print(f"\n  🍽️ پرتکرار:")
            for item, count in counter.most_common(5):
                print(f"     • {item}: {count}")


for brand in sorted(df_result["رستوران"].unique()):
    brand_report(df_result, brand)

In [ ]:
# @title
def comparison_table(df_result):
    rows = []
    for brand in df_result["رستوران"].unique():
        bdf = df_result[df_result["رستوران"] == brand]
        total = len(bdf)
        pos_pct = (bdf["احساسات"] == "مثبت").sum() / total * 100
        neg_pct = (bdf["احساسات"] == "منفی").sum() / total * 100
        avg_score = bdf["امتیاز_احساسات"].mean()
        avg_star = bdf["ستاره"].mean()

        aspect_avgs = {}
        for aspect in ASPECTS.keys():
            col = f"جنبه_{aspect}"
            if col in bdf.columns:
                vals = bdf[col].dropna()
                aspect_avgs[aspect] = round(vals.mean(), 2) if len(vals) > 0 else "-"

        rows.append({
            "برند": brand,
            "نظرات": total,
            "ستاره": round(avg_star, 2),
            "مثبت٪": round(pos_pct, 1),
            "منفی٪": round(neg_pct, 1),
            "امتیاز_احساسات": round(avg_score, 3),
            **aspect_avgs,
        })

    comp = pd.DataFrame(rows).sort_values("امتیاز_احساسات", ascending=False)
    comp.index = range(1, len(comp) + 1)
    comp.index.name = "رتبه"
    comp.to_excel("brand_comparison.xlsx")
    return comp

comp = comparison_table(df_result)
print("🏆 رتبه‌بندی برندها:\n")
comp

In [ ]:
# @title
import plotly.express as px
import plotly.graph_objects as go

# ۱. توزیع احساسات
fig = px.histogram(
    df_result, x="رستوران", color="احساسات", barmode="group",
    title="توزیع احساسات نظرات هر برند",
    color_discrete_map={"مثبت": "#2ecc71", "منفی": "#e74c3c", "خنثی": "#95a5a6"},
)
fig.update_layout(xaxis_title="برند", yaxis_title="تعداد نظر")
fig.show()

# ۲. هیت‌مپ جنبه‌ها
aspect_cols = [f"جنبه_{a}" for a in ASPECTS.keys()]
brand_aspects = df_result.groupby("رستوران")[aspect_cols].mean()
brand_aspects.columns = list(ASPECTS.keys())
fig = px.imshow(brand_aspects, text_auto=".2f",
    title="هیت‌مپ جنبه‌ای (سبز=خوب 🔴 قرمز=بد)",
    color_continuous_scale="RdYlGn", zmin=-1, zmax=1)
fig.show()

# ۳. رادار ۵ برند برتر
top5 = df_result.groupby("رستوران")["امتیاز_احساسات"].mean().nlargest(5)
fig = go.Figure()
for brand in top5.index:
    bdf = df_result[df_result["رستوران"] == brand]
    values = []
    for aspect in ASPECTS.keys():
        col = f"جنبه_{aspect}"
        vals = bdf[col].dropna()
        values.append(vals.mean() if len(vals) > 0 else 0)
    fig.add_trace(go.Scatterpolar(r=values, theta=list(ASPECTS.keys()),
        fill='toself', name=brand))
fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[-1, 1])),
    title="رادار ۵ برند برتر", showlegend=True)
fig.show()

# ۴. امتیاز کلی
brand_avg = df_result.groupby("رستوران")["امتیاز_احساسات"].mean().sort_values()
fig = px.bar(x=brand_avg.values, y=brand_avg.index, orientation='h',
    title="امتیاز کلی احساسات", color=brand_avg.values,
    color_continuous_scale="RdYlGn", range_color=[-1, 1])
fig.update_layout(xaxis_title="امتیاز", height=600)
fig.show()

# ۵. مقایسه ستاره با احساسات
brand_star = df_result.groupby("رستوران").agg(
    ستاره=("ستاره", "mean"),
    احساسات=("امتیاز_احساسات", "mean"),
).sort_values("احساسات", ascending=False)
fig = px.bar(brand_star, barmode="group",
    title="مقایسه ستاره کاربر با امتیاز احساسات متن")
fig.show()

In [ ]:
# @title
def find_gaps(df_result):
    print("\n" + "="*60)
    print("🎯 شکاف‌ها و فرصت‌های بازار")
    print("="*60)

    # ۱. جنبه‌های ضعیف کل بازار
    print("\n📍 امتیاز هر جنبه در کل بازار:")
    print("-"*45)
    aspect_avgs = {}
    for aspect in ASPECTS.keys():
        col = f"جنبه_{aspect}"
        vals = df_result[col].dropna()
        if len(vals) > 0:
            avg = vals.mean()
            aspect_avgs[aspect] = avg
            emoji = "🔴" if avg < 0 else ("🟡" if avg < 0.3 else "🟢")
            print(f"  {emoji} {aspect:>8}: {avg:+.3f}")

    weakest = min(aspect_avgs, key=aspect_avgs.get)
    print(f"\n  💡 بزرگترین فرصت: {weakest} → بازار اینجا ضعیفه")

    # ۲. نقطه ضعف هر برند
    print("\n\n📍 نقطه ضعف هر رقیب:")
    print("-"*45)
    for brand in df_result["رستوران"].unique():
        bdf = df_result[df_result["رستوران"] == brand]
        weakest_asp, weakest_sc = None, 999
        for aspect in ASPECTS.keys():
            col = f"جنبه_{aspect}"
            vals = bdf[col].dropna()
            if len(vals) > 0 and vals.mean() < weakest_sc:
                weakest_sc = vals.mean()
                weakest_asp = aspect
        if weakest_asp:
            print(f"  {brand:>15}: {weakest_asp} ({weakest_sc:+.3f})")

    # ۳. فرصت حمله
    print("\n\n📍 فرصت حمله (رقیب قوی با جنبه ضعیف):")
    print("-"*45)
    for brand in df_result["رستوران"].unique():
        bdf = df_result[df_result["رستوران"] == brand]
        overall = bdf["امتیاز_احساسات"].mean()
        if overall > 0.2:
            for aspect in ASPECTS.keys():
                col = f"جنبه_{aspect}"
                vals = bdf[col].dropna()
                if len(vals) > 0 and vals.mean() < -0.1:
                    print(f"  ⚔️ {brand}: کلی {overall:+.2f} ولی {aspect} ضعیف ({vals.mean():+.2f})")

find_gaps(df_result)

In [ ]:
# @title
# ============================================================
# 🎯 امتیاز وزنی کیفیت + نقشه قیمت-کیفیت
# ============================================================

QUALITY_WEIGHTS = {
    "غذا":       0.50,
    "ارسال":     0.30,
    "بسته‌بندی": 0.20,
}

def compute_quality_score(df_result):
    brands = sorted(df_result["رستوران"].unique())
    rows = []

    for brand in brands:
        bdf = df_result[df_result["رستوران"] == brand]
        total = len(bdf)
        if total == 0:
            continue

        aspect_scores = {}
        aspect_counts = {}
        for aspect in ["غذا", "ارسال", "بسته‌بندی"]:
            col = f"جنبه_{aspect}"
            if col in bdf.columns:
                vals = bdf[col].dropna()
                aspect_scores[aspect] = vals.mean() if len(vals) >= MIN_MENTIONS else None
                aspect_counts[aspect] = len(vals)
            else:
                aspect_scores[aspect] = None
                aspect_counts[aspect] = 0

        weighted_sum = 0
        weight_used = 0
        for aspect, weight in QUALITY_WEIGHTS.items():
            if aspect_scores.get(aspect) is not None:
                weighted_sum += weight * aspect_scores[aspect]
                weight_used += weight

        quality_score = weighted_sum / weight_used if weight_used > 0 else None

        overall_score = bdf["امتیاز_احساسات"].mean()
        avg_star = bdf["ستاره"].mean()

        price_row = df_price[df_price["رستوران"] == brand]
        avg_price = price_row["میانگین_قیمت"].values[0] if len(price_row) > 0 else None
        price_score = price_row["امتیاز_قیمت"].values[0] if len(price_row) > 0 else None

        rows.append({
            "رستوران": brand,
            "تعداد_نظرات": total,
            "ستاره_میانگین": round(avg_star, 2),
            "امتیاز_احساسات": round(overall_score, 3),
            "غذا_امتیاز": round(aspect_scores.get("غذا", 0) or 0, 3),
            "غذا_اشاره": aspect_counts.get("غذا", 0),
            "ارسال_امتیاز": round(aspect_scores.get("ارسال", 0) or 0, 3),
            "ارسال_اشاره": aspect_counts.get("ارسال", 0),
            "بسته‌بندی_امتیاز": round(aspect_scores.get("بسته‌بندی", 0) or 0, 3),
            "بسته‌بندی_اشاره": aspect_counts.get("بسته‌بندی", 0),
            "امتیاز_کیفیت": round(quality_score, 3) if quality_score is not None else None,
            "میانگین_قیمت": avg_price,
            "امتیاز_قیمت": price_score,
        })

    return pd.DataFrame(rows)


df_quality = compute_quality_score(df_result)

print("\n" + "="*70)
print("🏆 امتیاز کیفیت وزنی هر برند")
print("="*70)
print(f"وزن‌ها: غذا={QUALITY_WEIGHTS['غذا']}, ارسال={QUALITY_WEIGHTS['ارسال']}, بسته‌بندی={QUALITY_WEIGHTS['بسته‌بندی']}")
print()
print(df_quality[["رستوران","ستاره_میانگین","غذا_امتیاز","ارسال_امتیاز","بسته‌بندی_امتیاز","امتیاز_کیفیت","امتیاز_قیمت","میانگین_قیمت"]].to_string(index=False))


# ============================================================
# 🗺️ نقشه قیمت-کیفیت
# ============================================================

df_map = df_quality.dropna(subset=["امتیاز_کیفیت"]).copy()

# ===== جیتتر برای جلوگیری از هم‌پوشانی =====
np.random.seed(42)
jitter_x = np.random.uniform(-0.03, 0.03, len(df_map))
jitter_y = np.random.uniform(-0.03, 0.03, len(df_map))

if len(df_map) > 0 and df_map["امتیاز_قیمت"].notna().any():

    df_map["قیمت_نرمال"] = (df_map["امتیاز_قیمت"] - 3) / 2
    df_map["قیمت_جیتتر"] = df_map["قیمت_نرمال"] + jitter_x[:len(df_map)]
    df_map["کیفیت_جیتتر"] = df_map["امتیاز_کیفیت"] + jitter_y[:len(df_map)]

    # محاسبه range مناسب
    x_min = min(df_map["قیمت_نرمال"].min(), -0.8) - 0.4
    x_max = max(df_map["قیمت_نرمال"].max(), 0.8) + 0.4
    y_min = min(df_map["امتیاز_کیفیت"].min(), -0.5) - 0.3
    y_max = max(df_map["امتیاز_کیفیت"].max(), 0.5) + 0.3

    fig = go.Figure()

    # ناحیه‌ها (پس‌زمینه)
    fig.add_hrect(y0=0, y1=y_max+0.5, x0=0, x1=x_max+0.5,
                  fillcolor="rgba(46,204,113,0.06)", line_width=0)
    fig.add_hrect(y0=0, y1=y_max+0.5, x0=x_min-0.5, x1=0,
                  fillcolor="rgba(52,152,219,0.06)", line_width=0)
    fig.add_hrect(y0=y_min-0.5, y1=0, x0=0, x1=x_max+0.5,
                  fillcolor="rgba(243,156,18,0.06)", line_width=0)
    fig.add_hrect(y0=y_min-0.5, y1=0, x0=x_min-0.5, x1=0,
                  fillcolor="rgba(231,76,60,0.06)", line_width=0)

    # خطوط تقسیم
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)

    # برچسب ربع‌ها
    fig.add_annotation(x=x_max*0.65, y=y_max*0.85,
        text="⭐ طلایی<br>ارزان + کیفیت بالا",
        showarrow=False, font=dict(size=16, color="#27ae60", family="Vazir, Tahoma"))
    fig.add_annotation(x=x_min*0.65, y=y_max*0.85,
        text="💎 پریمیوم<br>گران + کیفیت بالا",
        showarrow=False, font=dict(size=16, color="#2980b9", family="Vazir, Tahoma"))
    fig.add_annotation(x=x_max*0.65, y=y_min*0.75,
        text="⚠️ اقتصادی ضعیف<br>ارزان + کیفیت پایین",
        showarrow=False, font=dict(size=16, color="#e67e22", family="Vazir, Tahoma"))
    fig.add_annotation(x=x_min*0.65, y=y_min*0.75,
        text="❌ بدترین<br>گران + کیفیت پایین",
        showarrow=False, font=dict(size=16, color="#c0392b", family="Vazir, Tahoma"))

    # حباب‌ها — با جیتتر
    fig.add_trace(go.Scatter(
        x=df_map["قیمت_جیتتر"],
        y=df_map["کیفیت_جیتتر"],
        mode="markers",
        marker=dict(
            size=[s * 12 + 15 for s in df_map["ستاره_میانگین"]],
            color=df_map["امتیاز_احساسات"],
            colorscale="RdYlGn",
            cmin=-0.5, cmax=0.5,
            line=dict(width=2.5, color="white"),
            showscale=True,
            colorbar=dict(title="رضایت", thickness=15, len=0.6),
            opacity=0.85,
        ),
        customdata=df_map[["رستوران","ستاره_میانگین","امتیاز_کیفیت","امتیاز_قیمت","تعداد_نظرات"]],
        hovertemplate=(
            "<b>%{customdata0}</b><br>"
            "ستاره: %{customdata1:.1f}<br>"
            "کیفیت: %{customdata2:.3f}<br>"
            "امتیاز قیمت: %{customdata3:.2f}<br>"
            "تعداد نظرات: %{customdata4}"
            "<extra></extra>"
        ),
        showlegend=False,
    ))

    # لیبل‌ها جداگانه — با آفست برای خوانایی
    fig.add_trace(go.Scatter(
        x=df_map["قیمت_جیتتر"],
        y=df_map["کیفیت_جیتتر"],
        mode="text",
        text=df_map["رستوران"],
        textposition="top center",
        textfont=dict(size=11, family="Vazir, Tahoma", color="#2c3e50"),
        showlegend=False,
    ))

    fig.update_layout(
        title=dict(
            text="🗺️ نقشه قیمت-کیفیت (اندازه حباب = ستاره | رنگ = رضایت)",
            font=dict(size=18)
        ),
        xaxis_title="← گرانتر  |  امتیاز قیمت  |  ارزانتر →",
        yaxis_title="← ضعیف‌تر  |  امتیاز کیفیت  |  بهتر →",
        xaxis_range=[x_min, x_max],
        yaxis_range=[y_min, y_max],
        height=800,
        width=1100,
        font=dict(family="Vazir, Tahoma, Arial", size=12),
        margin=dict(l=80, r=80, t=80, b=80),
    )
    fig.show()


elif len(df_map) > 0 and df_map["میانگین_قیمت"].notna().any():

    price_min = df_map["میانگین_قیمت"].min()
    price_max = df_map["میانگین_قیمت"].max()
    price_range = price_max - price_min if price_max > price_min else 1

    df_map["قیمت_نرمال"] = 1 - 2 * (df_map["میانگین_قیمت"] - price_min) / price_range
    df_map["قیمت_جیتتر"] = df_map["میانگین_قیمت"] + np.random.uniform(-price_range*0.02, price_range*0.02, len(df_map))
    df_map["کیفیت_جیتتر"] = df_map["امتیاز_کیفیت"] + jitter_y[:len(df_map)]

    x_pad = price_range * 0.15
    y_min_c = df_map["امتیاز_کیفیت"].min() - 0.3
    y_max_c = df_map["امتیاز_کیفیت"].max() + 0.3
    avg_price = df_map["میانگین_قیمت"].mean()

    fig = go.Figure()

    # ناحیه‌ها
    fig.add_hrect(y0=0, y1=y_max_c+0.5, x0=avg_price, x1=price_max+x_pad+0.5,
                  fillcolor="rgba(52,152,219,0.06)", line_width=0)
    fig.add_hrect(y0=0, y1=y_max_c+0.5, x0=price_min-x_pad-0.5, x1=avg_price,
                  fillcolor="rgba(46,204,113,0.06)", line_width=0)
    fig.add_hrect(y0=y_min_c-0.5, y1=0, x0=avg_price, x1=price_max+x_pad+0.5,
                  fillcolor="rgba(231,76,60,0.06)", line_width=0)
    fig.add_hrect(y0=y_min_c-0.5, y1=0, x0=price_min-x_pad-0.5, x1=avg_price,
                  fillcolor="rgba(243,156,18,0.06)", line_width=0)

    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
    fig.add_vline(x=avg_price, line_dash="dash", line_color="gray", line_width=1)

    fig.add_annotation(x=price_max*0.8, y=y_max_c*0.8,
        text="💎 گران + خوب", showarrow=False, font=dict(size=16, color="#2980b9", family="Vazir, Tahoma"))
    fig.add_annotation(x=price_min+price_range*0.1, y=y_max_c*0.8,
        text="⭐ ارزان + خوب", showarrow=False, font=dict(size=16, color="#27ae60", family="Vazir, Tahoma"))
    fig.add_annotation(x=price_max*0.8, y=y_min_c*0.7,
        text="❌ گران + ضعیف", showarrow=False, font=dict(size=16, color="#c0392b", family="Vazir, Tahoma"))
    fig.add_annotation(x=price_min+price_range*0.1, y=y_min_c*0.7,
        text="⚠️ ارزان + ضعیف", showarrow=False, font=dict(size=16, color="#e67e22", family="Vazir, Tahoma"))

    # حباب‌ها
    fig.add_trace(go.Scatter(
        x=df_map["قیمت_جیتتر"],
        y=df_map["کیفیت_جیتتر"],
        mode="markers",
        marker=dict(
            size=[s * 12 + 15 for s in df_map["ستاره_میانگین"]],
            color=df_map["امتیاز_احساسات"],
            colorscale="RdYlGn",
            cmin=-0.5, cmax=0.5,
            line=dict(width=2.5, color="white"),
            showscale=True,
            colorbar=dict(title="رضایت", thickness=15, len=0.6),
            opacity=0.85,
        ),
        customdata=df_map[["رستوران","ستاره_میانگین","امتیاز_کیفیت","میانگین_قیمت","تعداد_نظرات"]],
        hovertemplate=(
            "<b>%{customdata0}</b><br>"
            "ستاره: %{customdata1:.1f}<br>"
            "کیفیت: %{customdata2:.3f}<br>"
            "میانگین قیمت: %{customdata3:,.0f} تومان<br>"
            "تعداد نظرات: %{customdata4}"
            "<extra></extra>"
        ),
        showlegend=False,
    ))

    # لیبل‌ها
    fig.add_trace(go.Scatter(
        x=df_map["قیمت_جیتتر"],
        y=df_map["کیفیت_جیتتر"],
        mode="text",
        text=df_map["رستوران"],
        textposition="top center",
        textfont=dict(size=11, family="Vazir, Tahoma", color="#2c3e50"),
        showlegend=False,
    ))

    fig.update_layout(
        title=dict(
            text="🗺️ نقشه قیمت-کیفیت (اندازه حباب = ستاره | رنگ = رضایت)",
            font=dict(size=18)
        ),
        xaxis_title="میانگین قیمت (تومان) ←",
        yaxis_title="← ضعیف‌تر  |  امتیاز کیفیت  |  بهتر →",
        xaxis_range=[price_min - x_pad, price_max + x_pad],
        yaxis_range=[y_min_c, y_max_c],
        height=800,
        width=1100,
        font=dict(family="Vazir, Tahoma, Arial", size=12),
        margin=dict(l=80, r=80, t=80, b=80),
    )
    fig.show()

else:
    print("⚠️ داده قیمت موجود نیست — نقشه رسم نشد")


# ============================================================
# 📊 نمودار وزنی تفصیلی
# ============================================================

df_q = df_quality.dropna(subset=["امتیاز_کیفیت"]).sort_values("امتیاز_کیفیت", ascending=True)

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    y=df_q["رستوران"],
    x=df_q["امتیاز_کیفیت"],
    orientation='h',
    marker_color=df_q["امتیاز_کیفیت"],
    marker_colorscale="RdYlGn",
    marker_cmin=-0.5, marker_cmax=0.5,
    text=[f"{v:+.3f}" for v in df_q["امتیاز_کیفیت"]],
    textposition="outside",
    textfont=dict(size=9),
    name="امتیاز کیفیت",
))

fig2.add_vline(x=0, line_dash="solid", line_color="black", line_width=1)

fig2.update_layout(
    title=f"🏆 امتیاز کیفیت وزنی هر برند (غذا={QUALITY_WEIGHTS['غذا']}, ارسال={QUALITY_WEIGHTS['ارسال']}, بسته‌بندی={QUALITY_WEIGHTS['بسته‌بندی']})",
    xaxis_title="امتیاز کیفیت",
    height=max(500, len(df_q) * 35 + 150),
    font=dict(family="Vazir, Tahoma, Arial"),
)
fig2.show()


# ===== تجزیه کیفیت =====
fig3 = go.Figure()

aspects_for_chart = ["غذا_امتیاز", "ارسال_امتیاز", "بسته‌بندی_امتیاز"]
colors = {"غذا_امتیاز": "#e74c3c", "ارسال_امتیاز": "#3498db", "بسته‌بندی_امتیاز": "#1abc9c"}
labels = {"غذا_امتیاز": "غذا", "ارسال_امتیاز": "ارسال", "بسته‌بندی_امتیاز": "بسته‌بندی"}

for col in aspects_for_chart:
    fig3.add_trace(go.Bar(
        name=labels[col],
        y=df_q["رستوران"],
        x=df_q[col],
        orientation='h',
        marker_color=colors[col],
        text=[f"{v:+.2f}" for v in df_q[col]],
        textposition="outside",
        textfont=dict(size=8),
    ))

fig3.add_vline(x=0, line_dash="solid", line_color="black")
fig3.update_layout(
    barmode="group",
    title="📊 تجزیه امتیاز کیفیت: غذا vs ارسال vs بسته‌بندی",
    xaxis_title="امتیاز",
    height=max(500, len(df_q) * 35 + 200),
    font=dict(family="Vazir, Tahoma, Arial"),
)
fig3.show()


df_quality.to_excel("quality_scores.xlsx", index=False)
print("\n✅ فایل quality_scores.xlsx ذخیره شد")

In [ ]:
# @title
# ============================================================
# 🗺️ نقشه قیمت-کیفیت
# ============================================================

# ===== محاسبه امتیاز کیفیت هر برند =====
QUALITY_WEIGHTS = {"غذا": 0.50, "ارسال": 0.30, "بسته‌بندی": 0.20}

brands = sorted(df_result["رستوران"].unique())
quality_data = []

for brand in brands:
    bdf = df_result[df_result["رستوران"] == brand]
    if len(bdf) == 0:
        continue

    aspect_raw = {}
    for aspect in ["غذا", "ارسال", "بسته‌بندی"]:
        col = f"جنبه_{aspect}"
        if col in bdf.columns:
            vals = bdf[col].dropna()
            aspect_raw[aspect] = vals.mean() if len(vals) >= MIN_MENTIONS else None
        else:
            aspect_raw[aspect] = None

    w_sum = 0
    w_used = 0
    for asp, w in QUALITY_WEIGHTS.items():
        if aspect_raw.get(asp) is not None:
            w_sum += w * aspect_raw[asp]
            w_used += w

    if w_used > 0:
        quality_raw = w_sum / w_used
        quality_1to5 = round(3 + 2 * quality_raw, 2)
    else:
        quality_1to5 = None

    price_row = df_price[df_price["رستوران"] == brand]
    price_score = None
    if len(price_row) > 0:
        if "امتیاز_قیمت" in df_price.columns:
            price_score = pd.to_numeric(price_row["امتیاز_قیمت"].values[0], errors="coerce")
        elif len(df_price.columns) >= 3:
            price_score = pd.to_numeric(price_row.iloc[0, 2], errors="coerce")

    quality_data.append({
        "رستوران": brand,
        "کیفیت_۱تا۵": quality_1to5,
        "امتیاز_قیمت": price_score,
    })

df_map = pd.DataFrame(quality_data)
df_valid = df_map.dropna(subset=["کیفیت_۱تا۵", "امتیاز_قیمت"]).copy()

if len(df_valid) == 0:
    print("❌ داده کافی نیست")
    print(f"   ستون‌های تب قیمت: {df_price.columns.tolist()}")
    print(df_price.head())
else:
    # نرمال‌سازی قیمت به ۱-۵
    if df_valid["امتیاز_قیمت"].max() <= 5 and df_valid["امتیاز_قیمت"].min() >= 1:
        df_valid["قیمت_نرمال"] = df_valid["امتیاز_قیمت"]
    else:
        p_min = df_valid["امتیاز_قیمت"].min()
        p_max = df_valid["امتیاز_قیمت"].max()
        p_range = p_max - p_min if p_max > p_min else 1
        df_valid["قیمت_نرمال"] = 1 + 4 * (df_valid["امتیاز_قیمت"] - p_min) / p_range

    med_q = df_valid["کیفیت_۱تا۵"].median()
    med_p = df_valid["قیمت_نرمال"].median()

    # ===== جیتتر هوشمند — فاصله‌دهی نقاط نزدیک =====
    df_valid = df_valid.sort_values("کیفیت_۱تا۵", ascending=False).reset_index(drop=True)

    x_vals = df_valid["قیمت_نرمال"].values.copy()
    y_vals = df_valid["کیفیت_۱تا۵"].values.copy()

    for i in range(len(df_valid)):
        for j in range(i+1, len(df_valid)):
            dx = x_vals[i] - x_vals[j]
            dy = y_vals[i] - y_vals[j]
            dist = np.sqrt(dx**2 + dy**2)
            if dist < 0.35:
                if abs(dx) < abs(dy):
                    shift = (0.35 - dist) / 2
                    if y_vals[i] >= y_vals[j]:
                        y_vals[i] += shift
                        y_vals[j] -= shift
                    else:
                        y_vals[i] -= shift
                        y_vals[j] += shift
                else:
                    shift = (0.35 - dist) / 2
                    if x_vals[i] >= x_vals[j]:
                        x_vals[i] += shift
                        x_vals[j] -= shift
                    else:
                        x_vals[i] -= shift
                        x_vals[j] += shift

    df_valid["x_plot"] = x_vals
    df_valid["y_plot"] = y_vals

    # ===== جایگاه متن‌ها =====
    text_positions = []
    for idx, row in df_valid.iterrows():
        y = row["y_plot"]
        x = row["x_plot"]
        pos = "top center"
        if y > 4.7:
            pos = "bottom center"
        elif x > 4.5:
            pos = "middle left"
        elif x < 1.5:
            pos = "middle right"
        text_positions.append(pos)

    fig = go.Figure()

    # ۴ ربع
    fig.add_shape(type="rect", x0=1, y0=med_q, x1=med_p, y1=5,
                  fillcolor="rgba(46,204,113,0.07)", line_width=0)
    fig.add_shape(type="rect", x0=med_p, y0=med_q, x1=5, y1=5,
                  fillcolor="rgba(52,152,219,0.07)", line_width=0)
    fig.add_shape(type="rect", x0=1, y0=1, x1=med_p, y1=med_q,
                  fillcolor="rgba(243,156,18,0.07)", line_width=0)
    fig.add_shape(type="rect", x0=med_p, y0=1, x1=5, y1=med_q,
                  fillcolor="rgba(231,76,60,0.07)", line_width=0)

    # نقاط برندها
    fig.add_trace(go.Scatter(
        x=df_valid["x_plot"],
        y=df_valid["y_plot"],
        mode="markers+text",
        marker=dict(
            size=18,
            color=df_valid["کیفیت_۱تا۵"],
            colorscale="RdYlGn",
            cmin=1, cmax=5,
            line=dict(width=2.5, color="white"),
            showscale=True,
            colorbar=dict(title=dict(text="کیفیت", font=dict(size=13)),
                          thickness=18, tickfont=dict(size=11)),
        ),
        text=df_valid["رستوران"],
        textposition=text_positions,
        textfont=dict(size=11, family="Vazir, Tahoma"),
    ))

    # خطوط میانه
    fig.add_hline(y=med_q, line_dash="dash", line_color="gray", line_width=1,
                  annotation_text=f"میانه کیفیت: {med_q:.2f}",
                  annotation_position="top left",
                  annotation_font=dict(color="gray", size=10))
    fig.add_vline(x=med_p, line_dash="dash", line_color="gray", line_width=1,
                  annotation_text=f"میانه قیمت: {med_p:.2f}",
                  annotation_position="top left",
                  annotation_font=dict(color="gray", size=10))

    fig.update_layout(
        title=dict(
            text="🗺️ نقشه قیمت-کیفیت",
            font=dict(size=22),
        ),
        xaxis=dict(
            title=dict(text="امتیاز قیمت", font=dict(size=14)),
            range=[0.5, 5.5], dtick=1,
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            title=dict(text="امتیاز کیفیت", font=dict(size=14)),
            range=[0.5, 5.5], dtick=1,
            tickfont=dict(size=12),
        ),
        height=850,
        width=1200,
        font=dict(family="Vazir, Tahoma, Arial"),
        plot_bgcolor="white",
        margin=dict(t=80, b=60, l=70, r=100),
    )
    fig.show()

In [ ]:
# @title
# ============================================================
# 🎨 نمودار ۱: کارت گزارش هر برند (دشبورد حرفه‌ای — نسخه نهایی)
# ============================================================

def brand_dashboard(df_result, brand_name):
    bdf = df_result[df_result["رستوران"] == brand_name]
    if bdf.empty:
        print(f"❌ داده‌ای برای {brand_name} نیست")
        return

    total = len(bdf)
    pos = (bdf["احساسات"] == "مثبت").sum()
    neg = (bdf["احساسات"] == "منفی").sum()
    neu = (bdf["احساسات"] == "خنثی").sum()
    avg = bdf["امتیاز_احساسات"].mean()
    avg_star = bdf["ستاره"].mean()
    satisfaction_rate = round(pos / total * 100, 1) if total > 0 else 0

    star_color = "#27ae60" if avg_star >= 3.5 else "#f39c12" if avg_star >= 2.5 else "#e74c3c"

    fig = go.Figure()

    # =============================================
    # ۱. دونات احساسات — بالا چپ
    # =============================================
    fig.add_trace(go.Pie(
        labels=["مثبت", "منفی", "خنثی"],
        values=[pos, neg, neu],
        marker=dict(
            colors=["#2ecc71", "#e74c3c", "#95a5a6"],
            line=dict(color="#ffffff", width=3),
        ),
        hole=0.62,
        textinfo="percent",
        textfont=dict(size=12, family="Vazir, Tahoma", color="#2c3e50"),
        textposition="outside",
        pull=[0.02, 0.02, 0.01],
        showlegend=True,
        legendgroup="sentiment",
        direction="clockwise",
        sort=False,
        domain=dict(x=[0.03, 0.45], y=[0.57, 0.95]),
    ))

    # متن مرکز دونات
    fig.add_annotation(
        x=0.24, y=0.76,
        text=f"<b>{total}</b><br><span style='font-size:11px;color:#95a5a6'>نظر</span>",
        showarrow=False,
        font=dict(size=28, family="Vazir, Tahoma", color="#2c3e50"),
        xref="paper", yref="paper",
    )

    # =============================================
    # ۲. گیج ستاره — بالا راست
    # =============================================
    fig.add_trace(go.Indicator(
        mode="gauge+number+delta",
        value=avg_star,
        delta=dict(
            reference=3.0,
            decreasing=dict(color="#e74c3c"),
            increasing=dict(color="#2ecc71"),
            font=dict(size=14, family="Vazir, Tahoma"),
        ),
        number=dict(
            font=dict(size=36, family="Vazir, Tahoma", color=star_color),
            suffix=" / ۵",
        ),
        gauge=dict(
            axis=dict(
                range=[0, 5],
                tickwidth=1.5,
                tickcolor="#dee2e6",
                tickfont=dict(size=9, family="Vazir, Tahoma", color="#95a5a6"),
                dtick=1,
            ),
            bar=dict(color=star_color, thickness=0.35),
            bgcolor="#fafbfc",
            borderwidth=2,
            bordercolor="#dee2e6",
            steps=[
                dict(range=[0, 2], color="#fdedec"),
                dict(range=[2, 3.5], color="#fef9e7"),
                dict(range=[3.5, 5], color="#eafaf1"),
            ],
            threshold=dict(
                line=dict(color="#e74c3c", width=3),
                thickness=0.7,
                value=3.0,
            ),
        ),
        title=dict(text=""),
        domain=dict(x=[0.57, 0.97], y=[0.57, 0.95]),
    ))

    # =============================================
    # ۳. امتیاز جنبه‌ای — پایین چپ
    # =============================================
    aspect_names = []
    aspect_values = []
    aspect_counts = []
    for aspect in ASPECTS.keys():
        col = f"جنبه_{aspect}"
        vals = bdf[col].dropna()
        if len(vals) > 0:
            aspect_names.append(aspect)
            aspect_values.append(round(vals.mean(), 3))
            aspect_counts.append(len(vals))

    colors_asp = []
    for v in aspect_values:
        if v > 0.3:
            colors_asp.append("#27ae60")
        elif v > 0:
            colors_asp.append("#f39c12")
        elif v > -0.3:
            colors_asp.append("#e67e22")
        else:
            colors_asp.append("#c0392b")

    fig.add_trace(go.Bar(
        x=aspect_names,
        y=aspect_values,
        marker_color=colors_asp,
        marker_line_color="#ffffff",
        marker_line_width=2,
        text=[f"{v:+.2f}<br><span style='font-size:8px;color:#95a5a6'>({c})</span>" for v, c in zip(aspect_values, aspect_counts)],
        textposition="outside",
        textfont=dict(size=11, family="Vazir, Tahoma"),
        xaxis="x",
        yaxis="y",
    ))

    # =============================================
    # ۴. مثبت / منفی / خنثی — پایین راست
    # =============================================
    fig.add_trace(go.Bar(
        x=["مثبت", "منفی", "خنثی"],
        y=[pos, neg, neu],
        marker_color=["#2ecc71", "#e74c3c", "#95a5a6"],
        marker_line_color="#ffffff",
        marker_line_width=2,
        text=[f"<b>{v}</b>" for v in [pos, neg, neu]],
        textposition="outside",
        textfont=dict(size=14, family="Vazir, Tahoma", color="#2c3e50"),
        xaxis="x2",
        yaxis="y2",
    ))

    # =============================================
    # خط صفر نمودار جنبه‌ای
    # =============================================
    fig.add_shape(
        type="line",
        x0=-0.5, x1=len(aspect_names) - 0.5,
        y0=0, y1=0,
        xref="x", yref="y",
        line=dict(color="#bdc3c7", width=1.5),
    )

    # =============================================
    # کارت‌های پس‌زمینه
    # =============================================
    cards = [
        (0.02, 0.56, 0.46, 0.96, "#eafaf1", "#27ae60"),
        (0.54, 0.56, 0.98, 0.96, "#fef9e7", "#f1c40f"),
        (0.02, 0.04, 0.46, 0.48, "#f5f6fa", "#3498db"),
        (0.54, 0.04, 0.98, 0.48, "#fdedec", "#e74c3c"),
    ]
    for x0, y0, x1, y1, fc, lc in cards:
        fig.add_shape(
            type="rect",
            x0=x0, y0=y0, x1=x1, y1=y1,
            xref="paper", yref="paper",
            fillcolor=fc,
            line=dict(color=lc, width=1.5),
            opacity=0.25,
        )

    # =============================================
    # عنوان‌های بخش‌ها
    # =============================================
    fig.add_annotation(
        x=0.24, y=0.975, xref="paper", yref="paper",
        text="🍰 <b>توزیع احساسات</b>",
        showarrow=False,
        font=dict(size=13, family="Vazir, Tahoma", color="#2c3e50"),
    )
    fig.add_annotation(
        x=0.77, y=0.975, xref="paper", yref="paper",
        text="⭐ <b>میانگین ستاره</b>",
        showarrow=False,
        font=dict(size=13, family="Vazir, Tahoma", color="#2c3e50"),
    )
    fig.add_annotation(
        x=0.24, y=0.495, xref="paper", yref="paper",
        text="📊 <b>امتیاز جنبه‌ای</b> <span style='font-size:9px;color:#95a5a6'>(تعداد اشاره)</span>",
        showarrow=False,
        font=dict(size=13, family="Vazir, Tahoma", color="#2c3e50"),
    )
    fig.add_annotation(
        x=0.77, y=0.495, xref="paper", yref="paper",
        text="⚖️ <b>مقایسه مثبت / منفی / خنثی</b>",
        showarrow=False,
        font=dict(size=13, family="Vazir, Tahoma", color="#2c3e50"),
    )

    # =============================================
    # حاشیه کلی
    # =============================================
    fig.add_shape(
        type="rect",
        x0=0, y0=0, x1=1, y1=1,
        xref="paper", yref="paper",
        line=dict(color="#c8d6e5", width=2.5),
        fillcolor="rgba(0,0,0,0)",
    )

    # =============================================
    # لایه‌بندی نهایی
    # =============================================
    fig.update_layout(
        title=dict(
            text=(
                f"📋 <b>کارت گزارش: {brand_name}</b>"
                f"&nbsp;&nbsp;│&nbsp;&nbsp;"
                f"📝 {total} نظر"
                f"&nbsp;&nbsp;│&nbsp;&nbsp;"
                f"✅ رضایت {satisfaction_rate}%"
                f"&nbsp;&nbsp;│&nbsp;&nbsp;"
                f"💬 احساسات: {avg:+.3f}"
            ),
            font=dict(size=20, family="Vazir, Tahoma", color="#1a1a2e"),
            x=0.5,
            xanchor="center",
            pad=dict(t=15, b=5),
        ),
        xaxis=dict(
            domain=[0.07, 0.41],
            anchor="y",
            tickfont=dict(size=12, family="Vazir, Tahoma", color="#2c3e50"),
            showgrid=False,
        ),
        yaxis=dict(
            domain=[0.10, 0.43],
            anchor="x",
            range=[-1.15, 1.15],
            tickvals=[-1, -0.5, 0, 0.5, 1],
            gridcolor="#e8e8e8",
            gridwidth=0.5,
            zeroline=False,
            title_text="امتیاز",
            title_font=dict(size=9, family="Vazir, Tahoma", color="#95a5a6"),
            tickfont=dict(size=9, family="Vazir, Tahoma", color="#bdc3c7"),
        ),
        xaxis2=dict(
            domain=[0.60, 0.94],
            anchor="y2",
            tickfont=dict(size=13, family="Vazir, Tahoma", color="#2c3e50"),
            showgrid=False,
        ),
        yaxis2=dict(
            domain=[0.10, 0.43],
            anchor="x2",
            gridcolor="#e8e8e8",
            gridwidth=0.5,
            title_text="تعداد نظرات",
            title_font=dict(size=9, family="Vazir, Tahoma", color="#95a5a6"),
            tickfont=dict(size=9, family="Vazir, Tahoma", color="#bdc3c7"),
        ),
        height=800,
        width=1100,
        font=dict(family="Vazir, Tahoma, Arial"),
        plot_bgcolor="#ffffff",
        paper_bgcolor="#ffffff",
        margin=dict(t=105, b=25, l=50, r=35),
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="bottom",
            y=0.585,
            xanchor="left",
            x=0.025,
            font=dict(size=10, family="Vazir, Tahoma", color="#2c3e50"),
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor="#ecf0f1",
            borderwidth=1,
            traceorder="normal",
            itemsizing="constant",
            itemwidth=30,
        ),
    )

    fig.show()


for brand in sorted(df_result["رستوران"].unique()):
    brand_dashboard(df_result, brand)